In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
from keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout
from keras.models import Sequential
from keras.layers import BatchNormalization


In [3]:
from google.colab import drive
drive.mount('/content/drive')
path = 'drive/My Drive/dataset'
data = pd.read_csv(path + "/IoT Network Intrusion Dataset.csv",low_memory=False)

Mounted at /content/drive


In [4]:

col_in = []
for i in data.columns:
    unique_values = data[i].nunique()
    if unique_values == 1:
        col_in.append(i)
    else:
            continue

In [5]:
# Supprimer les colonnes inutiles
col_in = ['Fwd_Pkt_Len_Std', 'Bwd_Pkt_Len_Std', 'Flow_IAT_Std', 'Fwd_IAT_Tot', 'Fwd_IAT_Mean', 'Fwd_IAT_Std', 'Fwd_IAT_Max', 'Fwd_IAT_Min', 'Bwd_IAT_Tot', 'Bwd_IAT_Mean', 'Bwd_IAT_Std', 'Bwd_IAT_Max', 'Bwd_IAT_Min', 'Fwd_PSH_Flags', 'Bwd_PSH_Flags', 'Fwd_URG_Flags', 'Bwd_URG_Flags', 'Pkt_Len_Std', 'Pkt_Len_Var', 'FIN_Flag_Cnt', 'SYN_Flag_Cnt', 'RST_Flag_Cnt', 'PSH_Flag_Cnt', 'URG_Flag_Cnt', 'CWE_Flag_Count', 'ECE_Flag_Cnt', 'Down/Up_Ratio', 'Fwd_Byts/b_Avg', 'Fwd_Pkts/b_Avg', 'Fwd_Blk_Rate_Avg', 'Bwd_Byts/b_Avg', 'Bwd_Pkts/b_Avg', 'Bwd_Blk_Rate_Avg', 'Fwd_Seg_Size_Min', 'Active_Mean', 'Active_Std', 'Active_Max', 'Active_Min', 'Idle_Std']
data.drop(columns=col_in, inplace=True)

# Supprimer les colonnes non numériques
data.drop(columns=['Timestamp', 'Fwd_Pkt_Len_Mean', 'Bwd_Pkt_Len_Mean', 'Flow_IAT_Mean', 'Pkt_Len_Mean', 'Idle_Mean'], inplace=True)


In [6]:
#Remplacer les valeurs manquantes par 0

data.fillna(0)

,Flow_ID,Src_IP,Src_Port,Dst_IP,Dst_Port,Protocol,Flow_Duration,Tot_Fwd_Pkts,Tot_Bwd_Pkts,TotLen_Fwd_Pkts,...,Subflow_Bwd_Pkts,Subflow_Bwd_Byts,Init_Fwd_Win_Byts,Init_Bwd_Win_Byts,Fwd_Act_Data_Pkts,Idle_Max,Idle_Min,Label,Cat,Sub_Cat
0,192.168.0.13-192.168.0.16-10000-10101-17,192.168.0.13,10000,192.168.0.16,10101,17,75,1,1,982.0,...,1,1430,-1,-1,1,75.0,75.0,Anomaly,Mirai,Mirai-Ackflooding
1,192.168.0.13-222.160.179.132-554-2179-6,222.160.179.132,2179,192.168.0.13,554,6,5310,1,2,0.0,...,2,0,-1,14600,0,4254.0,1056.0,Anomaly,DoS,DoS-Synflooding
2,192.168.0.13-192.168.0.16-9020-52727-6,192.168.0.16,52727,192.168.0.13,9020,6,141,0,3,0.0,...,3,2806,-1,1869,0,71.0,70.0,Anomaly,Scan,Scan Port OS
3,192.168.0.13-192.168.0.16-9020-52964-6,192.168.0.16,52964,192.168.0.13,9020,6,151,0,2,0.0,...,2,2776,-1,1869,0,151.0,151.0,Anomaly,Mirai,Mirai-Hostbruteforceg
4,192.168.0.1-239.255.255.250-36763-1900-17,192.168.0.1,36763,239.255.255.250,1900,17,153,2,1,886.0,...,1,420,-1,-1,2,77.0,76.0,Anomaly,Mirai,Mirai-Hostbruteforceg
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
625778,192.168.0.24-210.89.164.90-56112-8043-17,192.168.0.24,56112,210.89.164.90,8043,17,277,1,1,18.0,...,1,18,-1,-1,1,277.0,277.0,Anomaly,Mirai,Mirai-UDP Flooding
625779,192.168.0.13-222.131.171.244-554-4570-6,222.131.171.244,4570,192.168.0.13,554,6,1658,0,2,0.0,...,2,0,-1,14600,0,1658.0,1658.0,Anomaly,DoS,DoS-Synflooding
625780,192.168.0.13-192.168.0.16-9020-52739-6,192.168.0.16,52739,192.168.0.13,9020,6,77,1,1,0.0,...,1,0,-1,32679,0,77.0,77.0,Anomaly,Scan,Scan Port OS
625781,192.168.0.13-192.168.0.16-9020-49784-6,192.168.0.13,9020,192.168.0.16,49784,6,240,2,1,2776.0,...,1,1388,-1,1869,2,125.0,115.0,Normal,Normal,Normal


In [7]:
#Supprimer les informations dupliquées
data = data.drop_duplicates(keep='first')

In [8]:
# Séparer les caractéristiques et les étiquettes
X = data.drop(columns=['Label', 'Cat', 'Sub_Cat'])
y = data['Label']

In [9]:
from sklearn.preprocessing import OrdinalEncoder

# Encoder les colonnes catégoriques
encoder = OrdinalEncoder()
cat_cols = ['Flow_ID','Src_IP','Dst_IP']
X[cat_cols] = encoder.fit_transform(X[cat_cols])

In [10]:
# Identifier les valeurs infinies
contains_infinity = np.any(np.isinf(X))

if contains_infinity:
    # Remplacer les valeurs infinies par des NaN (pour les exclure de la normalisation)
    X[np.isinf(X)] = np.nan

    # Remplacer les NaN par une valeur appropriée (par exemple, 0)
    X = np.nan_to_num(X, nan=0.0)

In [11]:
encoder = LabelEncoder()
encoder.fit(y)
y= encoder.transform(y)

In [12]:
# Normalisation des caractéristiques
scaler = MinMaxScaler()
X = scaler.fit_transform(X)

In [13]:
#Convertir les champs en "float64"
X = X.astype(np.float64)

In [14]:
#Convertir les étiquettes en "float64"
y = y.astype(np.float64)

In [15]:
y# Diviser les données en ensembles de formation et de test
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [16]:
from keras.layers import BatchNormalization
def model():
    model = Sequential()
    model.add(Conv1D(75, kernel_size=3, strides=1, padding="same", activation="relu",  input_shape=(x_train.shape[1],1)))
    model.add(BatchNormalization())
    model.add(MaxPooling1D(2, strides = 2, padding="same"))
    model.add(Conv1D(50, kernel_size=3, strides=1, padding="same", activation="relu"))
    model.add(Dropout(0.2))
    model.add(BatchNormalization())
    model.add(MaxPooling1D(2, strides=2, padding="same"))
    model.add(Conv1D(25, kernel_size=3, strides=1, padding="same", activation="relu"))
    model.add(BatchNormalization())
    model.add(MaxPooling1D(2, strides=2, padding="same"))
    model.add(Flatten())
    model.add(Dense(units=512, activation="relu"))
    model.add(Dropout(0.3))
    model.add(Dense(units=1, activation="sigmoid"))

    model.compile(optimizer='adam',loss="binary_crossentropy", metrics=["accuracy"])

    return model

In [17]:
!pip install scikeras
from scikeras.wrappers import KerasClassifier

# create the classifier
classifier = KerasClassifier(model)

In [18]:
# Choisir les données d'entraînement initiales de l'apprentissage actif de manière aléatoire
n_initial = 50000
initial_idx = np.random.choice(range(len(x_train)), size=n_initial, replace=False)
x_initial = x_train[initial_idx]
y_initial = y_train[initial_idx]

# Supprimer les données initiales du reste des données d'apprentissage
x_pool, y_pool = np.delete(x_train, initial_idx, axis=0), np.delete(y_train, initial_idx, axis=0)
#x_pool_test, y_pool_test = np.delete(x_test, initial_idx, axis=0), np.delete(y_test, initial_idx, axis=0)


In [19]:
x_test.shape

(61254, 38)

In [20]:
x_initial[1]

array([2.12228587e-01, 4.46381760e-01, 9.16839695e-01, 4.88469602e-01,
       1.36130700e-01, 1.00000000e+00, 1.04016643e-03, 5.91397849e-02,
       0.00000000e+00, 3.20448628e-03, 4.13819502e-05, 2.18579235e-02,
       2.18579235e-02, 2.18579235e-02, 2.19178082e-02, 9.08093382e-04,
       2.30769231e-02, 2.60070219e-04, 5.00135036e-05, 2.29645094e-02,
       4.46428571e-04, 2.64423077e-02, 9.61538462e-03, 2.19178082e-02,
       2.18579235e-02, 0.00000000e+00, 1.58295282e-02, 2.18579235e-02,
       2.19178082e-02, 5.91397849e-02, 3.20448628e-03, 0.00000000e+00,
       4.13819502e-05, 0.00000000e+00, 0.00000000e+00, 5.91397849e-02,
       2.60070219e-04, 5.00135036e-05])

In [21]:
!pip install git+https://github.com/modAL-python/modAL.git

from modAL.uncertainty import uncertainty_sampling
from modAL.models import ActiveLearner

# Initialisation de ActiveLearner
learner = ActiveLearner(
    estimator=classifier,
    query_strategy=uncertainty_sampling,
    X_training=x_initial,
    y_training=y_initial,
    verbose=1
)

  Cloning https://github.com/modAL-python/modAL.git to /tmp/pip-req-build-8ugp39cl
  Running command git clone --filter=blob:none --quiet https://github.com/modAL-python/modAL.git /tmp/pip-req-build-8ugp39cl
  Resolved https://github.com/modAL-python/modAL.git to commit bba6f6fd00dbb862b1e09259b78caf6cffa2e755
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.8/125.8 kB 2.4 MB/s eta 0:00:00
  Created wheel for modAL-python: filename=modAL_python-0.4.2-py3-none-any.whl size=32646 sha256=050d6ccbe1b0e254fb7ccfdd31ff6626e31dac681c7e40a416e2b38b932d3c4c
  Stored in directory: /tmp/pip-ephem-wheel-cache-26p3frcl/wheels/5a/f4/3d/82862c8f8da3e309feceabed046d87b2cd414bf11515b9061c
Successfully built modAL-python


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1563/1563 ━━━━━━━━━━━━━━━━━━━━ 21s 11ms/step - accuracy: 0.9614 - loss: 0.1086


In [ ]:
from IPython import display
from matplotlib import pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import numpy as np

# Initialisez des listes pour stocker les métriques à chaque itération
accuracy_scores = []
precision_scores = []
recall_scores = []
f1_scores = []
losses = []
confusion_matrices = []
# La boucle d'apprentissage actif
n_queries = 10
for idx in range(n_queries):
    print('Query no. %d' % (idx+1))
    query_idx, query_instance = learner.query(x_pool, n_instances=100)
    learner.teach(x_pool[query_idx], y_pool[query_idx], only_new=False)
    # remove queried instance from pool
    x_pool = np.delete(x_pool, query_idx, axis=0)
    y_pool = np.delete(y_pool, query_idx)
    # Évaluez le modèle actuel sur un ensemble de validation (supposons que vous ayez un X_val et un y_val)
    y_pred = learner.predict(x_test)

    # Calculez et enregistrez les métriques d'apprentissage à chaque itération
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    # Calculez la perte (loss) du modèle sur l'ensemble de validation
    loss = learner.score(x_test, y_test)
    # Calculez la matrice de confusion
    confusion = confusion_matrix(y_test, y_pred)


    accuracy_scores.append(accuracy)
    precision_scores.append(precision)
    recall_scores.append(recall)
    f1_scores.append(f1)
    losses.append(loss)
    confusion_matrices.append(confusion)


Query no. 1
6095/6095 ━━━━━━━━━━━━━━━━━━━━ 36s 6ms/step


/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1566/1566 ━━━━━━━━━━━━━━━━━━━━ 20s 11ms/step - accuracy: 0.9588 - loss: 0.1152
1915/1915 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step
1915/1915 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step
Query no. 2
6092/6092 ━━━━━━━━━━━━━━━━━━━━ 18s 3ms/step


/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1569/1569 ━━━━━━━━━━━━━━━━━━━━ 23s 12ms/step - accuracy: 0.9601 - loss: 0.1111
1915/1915 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step
1915/1915 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step
Query no. 3
6088/6088 ━━━━━━━━━━━━━━━━━━━━ 19s 3ms/step


...

1575/1575 ━━━━━━━━━━━━━━━━━━━━ 22s 11ms/step - accuracy: 0.9584 - loss: 0.1134
1915/1915 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step
1915/1915 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step
Query no. 5
6082/6082 ━━━━━━━━━━━━━━━━━━━━ 18s 3ms/step


/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1579/1579 ━━━━━━━━━━━━━━━━━━━━ 22s 12ms/step - accuracy: 0.9568 - loss: 0.1172
1915/1915 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step
1915/1915 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step
Query no. 6
6079/6079 ━━━━━━━━━━━━━━━━━━━━ 18s 3ms/step


/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1582/1582 ━━━━━━━━━━━━━━━━━━━━ 22s 12ms/step - accuracy: 0.9589 - loss: 0.1119
1915/1915 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step
1564/1915 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step

In [ ]:
loss_list = [0.1113, 0.1142, 0.0883, 0.0753, 0.0653, 0.0947, 0.1118, 0.0634, 0.0420, 0.0719]

In [ ]:
y_pred

In [ ]:
accuracy_scores

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
# Définir le point de départ pour l'axe y
start_y = 0.901# Remplacez cette valeur par celle que vous souhaitez

fig, ax = plt.subplots(figsize=(8.5, 6), dpi=130)

ax.plot(accuracy_scores)
ax.scatter(range(10), accuracy_scores, s=13)

ax.xaxis.set_major_locator(mpl.ticker.MaxNLocator(nbins=5))
ax.yaxis.set_major_locator(mpl.ticker.MaxNLocator(nbins=20))
ax.yaxis.set_major_formatter(mpl.ticker.PercentFormatter(xmax=1))

# Définir la limite inférieure de l'axe y
ax.set_ylim(bottom=start_y, top=1)

ax.grid(True)


ax.set_xlabel('Numbre epoch')
ax.set_ylabel('Accuracy')

plt.show()

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
# Définir le point de départ pour l'axe y
start_y = 0.0001# Remplacez cette valeur par celle que vous souhaitez

fig, ax = plt.subplots(figsize=(8.5, 6), dpi=130)

ax.plot(loss_list)

ax.grid(True)


ax.set_xlabel('Numbre epoch ')
ax.set_ylabel('Loss')
plt.show()


In [ ]:
recall_scores

In [ ]:
f1_scores

In [ ]:
confusion_matrices

In [ ]:
import seaborn as sns


# Dessiner la matrice de confusion sous forme de heatmap
plt.figure(figsize=(8, 6))
sns.set(font_scale=1.2)
sns.heatmap(confusion_matrices[8], annot=True, fmt="d", cmap="Blues", cbar=True, annot_kws={"size":14})

plt.xlabel('Prédiction')
plt.ylabel('Label')

plt.show()
